# Taller 2 – Redes Neuronales Convolucionales, Transfer Learning y Data Augmentation

**Suficiencia – Introducción al Aprendizaje Profundo**
**Estudiante:** Diego Alexander Valencia Calderón
**Dataset:** Oxford-IIIT Pet (37 razas de perros y gatos)

**Objetivo:** clasificar la raza de una mascota con redes convolucionales y medir, con cuatro experimentos comparables, el efecto del *Transfer Learning*, del *fine-tuning* y del *Data Augmentation*. La meta es una exactitud (*accuracy*) mínima del 90 % en el conjunto de prueba con el mejor experimento.

> **Entorno recomendado:** Google Colab con GPU T4. En CPU el entrenamiento de los cuatro experimentos puede tardar varias horas.

| Sección | Contenido | Requisito |
|---|---|---|
| 1 | Preparación, análisis y procesamiento de los datos | a |
| 2 | Construcción de los modelos con EfficientNetB0 preentrenada | b |
| 3 | Funciones comunes de entrenamiento y evaluación | c |
| 4 | Experimento A: CNN desde cero (línea base) | c |
| 5 | Experimento B: Transfer Learning con backbone congelado | c |
| 6 | Experimento C: Transfer Learning + fine-tuning | c |
| 7 | Data Augmentation y Experimento D: fine-tuning + DA | c, d |
| 8 | Comparación de experimentos | e |
| 9 | Análisis técnico | f |
| 10 | Resumen de métricas y red de seguridad | — |

**Salidas generadas**

| Archivo | Contenido |
|---|---|
| `figuras/01_distribucion_clases.png` | Distribución de clases por conjunto |
| `figuras/02_ejemplos_dataset.png` | Ejemplos del dataset con su raza |
| `figuras/03_data_augmentation.png` | Efecto del Data Augmentation sobre una imagen |
| `figuras/04_curvas_entrenamiento.png` | Curvas de accuracy y loss de los 4 experimentos |
| `figuras/05_comparacion_experimentos.png` | Accuracy de prueba de los 4 experimentos |
| `figuras/06_matriz_confusion_mejor.png` | Matriz de confusión del mejor modelo |
| `modelos/t2_exp*.keras` | Mejor modelo de cada experimento |
| `resultados/history_exp*.csv` | Historial de entrenamiento de cada experimento |
| `resultados/tabla_comparativa.csv` | Tabla comparativa de los experimentos |
| `resultados/classification_report_mejor.txt` | Precision, recall y F1 por clase del mejor modelo |
| `resultados/metricas_taller2.json` | Resumen de métricas del taller |

## 0. Configuración del entorno

Detecta si el notebook corre en Colab o en local, define las rutas de salida, fija la semilla para reproducibilidad y verifica la GPU (indispensable en este taller).

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
NOTEBOOK = "taller2_cnn_tl_da/taller2_cnn_tl_da.ipynb"  # ruta de este notebook dentro del proyecto

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Busca la carpeta del proyecto en "Mi unidad" (hasta 4 niveles), sin importar dónde se haya subido
    MI_UNIDAD = Path("/content/drive/MyDrive")
    REPO_DIR = next(
        (
            ruta.parents[1]
            for nivel in ("*", "*/*", "*/*/*", "*/*/*/*")
            for ruta in MI_UNIDAD.glob(f"{nivel}/{NOTEBOOK}")
        ),
        MI_UNIDAD / "Suficiencia_Aprendizaje_Profundo",
    )
else:
    # En local se sube desde el directorio actual hasta encontrar la carpeta que contiene este notebook
    REPO_DIR = next(
        (carpeta for carpeta in (Path.cwd(), *Path.cwd().parents) if (carpeta / NOTEBOOK).exists()),
        Path.cwd().parent,
    )

assert REPO_DIR.exists(), f"No se encontró la carpeta del proyecto en {REPO_DIR}"

TALLER_DIR = REPO_DIR / "taller2_cnn_tl_da"
FIG_DIR = TALLER_DIR / "figuras"
RES_DIR = TALLER_DIR / "resultados"
MOD_DIR = TALLER_DIR / "modelos"
for carpeta in (FIG_DIR, RES_DIR, MOD_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)  # fija las semillas de Python, NumPy y TensorFlow

print("Entorno:", "Google Colab" if EN_COLAB else "Local")
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "no disponible")
print("Repositorio:", REPO_DIR)

## 1. Preparación, análisis y procesamiento de los datos (a)

### 1.1 Parámetros generales y red de seguridad

La variable `OBJETIVO` define la tarea:

- `"label"`: clasificación de las **37 razas** (tarea principal).
- `"species"`: clasificación binaria **perro / gato** (red de seguridad).

Si el mejor experimento no alcanza el 90 % en las 37 razas, basta con cambiar `OBJETIVO = "species"` en esta celda y ejecutar todo de nuevo. El número de clases, los nombres, la capa de salida y todas las figuras se ajustan solos. La partición train / val / test es la misma en ambos casos, porque se estratifica siempre por raza.

La celda también instala `importlib_resources` si falta: `tensorflow_datasets` lo necesita para descargar el dataset y algunas versiones de Colab no lo incluyen.

In [ ]:
import importlib
import importlib.util
import subprocess

# tensorflow_datasets necesita el paquete importlib_resources al descargar el dataset,
# pero algunas imágenes de Colab (Python 3.13) no lo traen instalado: se instala solo si falta
if importlib.util.find_spec("importlib_resources") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "importlib_resources"], check=True)
    importlib.invalidate_caches()

from tensorflow.keras.applications import efficientnet

# Tarea de clasificación: "label" = 37 razas | "species" = perro/gato (red de seguridad)
OBJETIVO = "label"

TAM_IMAGEN = 224
FORMA_ENTRADA = (TAM_IMAGEN, TAM_IMAGEN, 3)
TAMANO_BATCH = 32
PROPORCION_VAL = 0.15
META_ACCURACY = 0.90
AUTOTUNE = tf.data.AUTOTUNE

print(f"Tarea: {OBJETIVO} | imágenes {TAM_IMAGEN}x{TAM_IMAGEN} | batch {TAMANO_BATCH}")

### 1.2 Carga del dataset

Oxford-IIIT Pet contiene fotografías de 37 razas (12 de gatos y 25 de perros). Cada ejemplo incluye la imagen, la raza (`label`), la especie (`species`), la máscara de segmentación y el recuadro de la cabeza. `tensorflow_datasets` lo descarga la primera vez (unos 800 MB) y lo entrega dividido en `train` (3.680 imágenes) y `test` (3.669).

Se usa `shuffle_files=False` para que el orden de lectura sea siempre el mismo, condición necesaria para que la partición de validación sea reproducible.

In [ ]:
(ds_train_completo, ds_test_crudo), info = tfds.load(
    "oxford_iiit_pet",
    split=["train", "test"],
    with_info=True,
    shuffle_files=False,
)

NOMBRES_RAZAS = info.features["label"].names
NOMBRES_CLASES = info.features[OBJETIVO].names
NUM_CLASES = len(NOMBRES_CLASES)
ETIQUETAS_LEGIBLES = [nombre.replace("_", " ") for nombre in NOMBRES_CLASES]

print("Versión del dataset:", info.version)
print("Splits originales:", {nombre: split.num_examples for nombre, split in info.splits.items()})
print(f"Número de clases de la tarea: {NUM_CLASES}")
print(info.features)

### 1.3 Partición en entrenamiento, validación y prueba

| Conjunto | Origen | Proporción | Uso |
|---|---|---|---|
| Entrenamiento | Split `train` | 85 % | Ajustar los pesos |
| Validación | Split `train` | 15 % | *Early stopping*, selección del mejor modelo de cada experimento y elección del mejor experimento |
| Prueba | Split `test` completo | — | Evaluación final, sin intervenir en ninguna decisión |

Para estratificar se leen primero solo las etiquetas, sin decodificar las imágenes, lo que hace la lectura muy rápida. Luego `train_test_split` elige los índices con semilla 42 y estratificación por raza, y el dataset se filtra por posición.

In [ ]:
from sklearn.model_selection import train_test_split

# Lectura rápida de etiquetas: se omite la decodificación de imágenes y máscaras
DECODIFICADORES = {"image": tfds.decode.SkipDecoding(), "segmentation_mask": tfds.decode.SkipDecoding()}


def leer_etiquetas(split):
    ds = tfds.load("oxford_iiit_pet", split=split, decoders=DECODIFICADORES, shuffle_files=False)
    return np.array([(int(ej["label"]), int(ej["species"])) for ej in tfds.as_numpy(ds)])


etiquetas_train_completo = leer_etiquetas("train")  # columnas: [raza, especie]
etiquetas_test = leer_etiquetas("test")
COLUMNA = 0 if OBJETIVO == "label" else 1

indices = np.arange(len(etiquetas_train_completo))
idx_train, idx_val = train_test_split(
    indices,
    test_size=PROPORCION_VAL,
    stratify=etiquetas_train_completo[:, 0],  # siempre por raza: también queda estratificado por especie
    random_state=SEMILLA,
)
mascara_train = tf.constant(np.isin(indices, idx_train))


def filtrar(ds, en_train):
    # Conserva los ejemplos cuya posición pertenece (o no) a la partición de entrenamiento
    return (
        ds.enumerate()
        .filter(lambda i, ejemplo: tf.equal(tf.gather(mascara_train, i), en_train))
        .map(lambda i, ejemplo: ejemplo)
    )


ds_train_crudo = filtrar(ds_train_completo, True)
ds_val_crudo = filtrar(ds_train_completo, False)

y_train_np = etiquetas_train_completo[np.sort(idx_train), COLUMNA]
y_val_np = etiquetas_train_completo[np.sort(idx_val), COLUMNA]
y_test_np = etiquetas_test[:, COLUMNA]

resumen = pd.DataFrame(
    {"Imágenes": [len(y_train_np), len(y_val_np), len(y_test_np)]},
    index=pd.Index(["Entrenamiento", "Validación", "Prueba"], name="Conjunto"),
)
resumen["Porcentaje (%)"] = (resumen["Imágenes"] / resumen["Imágenes"].sum() * 100).round(2)
display(resumen)

### 1.4 Análisis exploratorio (EDA)

**Distribución de clases.** Se revisa si las clases están balanceadas en los tres conjuntos.

In [ ]:
conteos = pd.DataFrame(
    {
        "Entrenamiento": np.bincount(y_train_np, minlength=NUM_CLASES),
        "Validación": np.bincount(y_val_np, minlength=NUM_CLASES),
        "Prueba": np.bincount(y_test_np, minlength=NUM_CLASES),
    },
    index=pd.Index(ETIQUETAS_LEGIBLES, name="Clase"),
)
conteos["Total"] = conteos.sum(axis=1)
display(conteos.sort_values("Total"))
print("Mínimo / máximo de imágenes por clase en entrenamiento:", conteos["Entrenamiento"].min(), "/", conteos["Entrenamiento"].max())

orden = conteos.sort_values("Total").index
alto = max(4, 0.32 * NUM_CLASES)
fig, ax = plt.subplots(figsize=(11, alto))
izquierda = np.zeros(NUM_CLASES)
for conjunto, color in [("Entrenamiento", "steelblue"), ("Validación", "darkorange"), ("Prueba", "seagreen")]:
    valores = conteos.loc[orden, conjunto].values
    ax.barh(orden, valores, left=izquierda, color=color, label=conjunto)
    izquierda += valores
ax.set_xlabel("Número de imágenes")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, frameon=False)
ax.set_title(f"Distribución de clases por conjunto ({NUM_CLASES} clases)", pad=28)
ax.margins(y=0.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_distribucion_clases.png", dpi=150, bbox_inches="tight")
plt.show()

especies = pd.Series(etiquetas_train_completo[:, 1]).map(dict(enumerate(info.features["species"].names)))
print("Imágenes del split train por especie:", especies.value_counts().to_dict())

**Tamaño de las imágenes.** Las fotografías vienen con tamaños y proporciones distintos, así que hay que llevarlas a una resolución común antes de entrar a la red.

In [ ]:
tamanos = np.array([ej["image"].shape[:2] for ej in tfds.as_numpy(ds_train_crudo.take(500))])
estadisticas = pd.DataFrame(tamanos, columns=["Alto", "Ancho"]).describe().round(1)
print("Tamaño de una muestra de 500 imágenes de entrenamiento (píxeles):")
display(estadisticas)

**Ejemplos del dataset.** Una imagen por clase (o varias por clase en el modo perro / gato), con su nombre.

In [ ]:
N_GRILLA = 24
por_clase = int(np.ceil(N_GRILLA / NUM_CLASES))
ejemplos = {}
for ejemplo in tfds.as_numpy(ds_train_crudo):
    clase = int(ejemplo[OBJETIVO])
    lista = ejemplos.setdefault(clase, [])
    if len(lista) < por_clase:
        lista.append(ejemplo["image"])
    if len(ejemplos) == NUM_CLASES and all(len(v) >= por_clase for v in ejemplos.values()):
        break

seleccion = [(clase, imagen) for clase in sorted(ejemplos) for imagen in ejemplos[clase]][:N_GRILLA]
fig, axes = plt.subplots(4, 6, figsize=(15, 11))
for ax in axes.flat:
    ax.axis("off")
for ax, (clase, imagen) in zip(axes.flat, seleccion):
    ax.imshow(imagen)
    ax.set_title(ETIQUETAS_LEGIBLES[clase], fontsize=10)
fig.suptitle("Ejemplos del conjunto de entrenamiento", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_ejemplos_dataset.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.5 Pipeline de preprocesamiento

1. **Redimensionado a 224 × 224**, la resolución con la que se entrenó EfficientNetB0 en ImageNet.
2. **Caché en memoria.** Las imágenes redimensionadas se guardan como `uint8`, unos 1,1 GB para los tres conjuntos, así que la decodificación y el redimensionado se hacen una sola vez y no en cada época.
3. **Barajado** solo del entrenamiento, con semilla fija.
4. **Lotes de 32 imágenes.**
5. **`efficientnet.preprocess_input`.** En EfficientNet esta función es prácticamente la identidad, porque el modelo de Keras ya incluye internamente el reescalado y la normalización. Por eso la red recibe los píxeles en el rango **[0, 255]** y **no** se aplica un `Rescaling(1/255)` manual: hacerlo dejaría las imágenes casi negras para el backbone.
6. **`prefetch`**, para preparar el siguiente lote mientras la GPU entrena con el actual.

In [ ]:
def redimensionar(ejemplo):
    imagen = tf.image.resize(ejemplo["image"], (TAM_IMAGEN, TAM_IMAGEN))
    imagen = tf.cast(tf.clip_by_value(tf.round(imagen), 0, 255), tf.uint8)  # uint8 ocupa 4 veces menos en caché
    return imagen, ejemplo[OBJETIVO]


def preprocesar(imagenes, etiquetas):
    return efficientnet.preprocess_input(tf.cast(imagenes, tf.float32)), etiquetas


def crear_pipeline(ds, entrenamiento=False):
    ds = ds.map(redimensionar, num_parallel_calls=AUTOTUNE).cache()
    if entrenamiento:
        ds = ds.shuffle(len(idx_train), seed=SEMILLA, reshuffle_each_iteration=True)
    return ds.batch(TAMANO_BATCH).map(preprocesar, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


ds_train = crear_pipeline(ds_train_crudo, entrenamiento=True)
ds_val = crear_pipeline(ds_val_crudo)
ds_test = crear_pipeline(ds_test_crudo)

# Comprobaciones: forma, rango y orden de las etiquetas de prueba (se usa más adelante en la matriz de confusión)
imagenes, etiquetas = next(iter(ds_val))
print("Lote:", imagenes.shape, imagenes.dtype, "| rango:", float(tf.reduce_min(imagenes)), "-", float(tf.reduce_max(imagenes)))
y_test = np.concatenate([y.numpy() for _, y in ds_test])
assert np.array_equal(y_test, y_test_np), "El orden del conjunto de prueba no coincide"
print("Etiquetas de prueba verificadas:", len(y_test))

## 2. Construcción de los modelos con una arquitectura convolucional preentrenada (b)

**Backbone: EfficientNetB0.** Es una CNN diseñada con *compound scaling*: escala de forma equilibrada la profundidad, el ancho y la resolución. Está construida con bloques MBConv (convoluciones *depthwise separable* con *squeeze-and-excitation*). Con unos 4 millones de parámetros sin la cabeza de clasificación, alcanza alrededor de 77 % de *top-1* en ImageNet, una muy buena relación entre precisión y costo para la GPU de Colab.

**Cabeza de clasificación:**

| Capa | Función |
|---|---|
| `GlobalAveragePooling2D` | Resume cada uno de los 1.280 mapas de características en un valor; menos parámetros que `Flatten` y menos sobreajuste |
| `Dropout(0,3)` | Regularización de la cabeza |
| `Dense(NUM_CLASES, softmax)` | Probabilidad de cada clase |

Una sola función construye los modelos de los cuatro experimentos. Así, todos comparten exactamente la misma arquitectura y solo cambia lo que se quiere estudiar: pesos aleatorios o de ImageNet, backbone congelado o parcialmente entrenable, y con o sin Data Augmentation.

Con pesos de ImageNet, el backbone se llama con `training=False`. Así las capas *BatchNormalization* conservan las estadísticas aprendidas en ImageNet incluso durante el *fine-tuning*. Si se actualizaran con lotes de 32 imágenes, se destruiría parte de lo aprendido.

In [ ]:
DROPOUT_CABEZA = 0.3


def construir_modelo(nombre, pesos="imagenet", base_entrenable=False, aumentacion=None):
    # Construye EfficientNetB0 + cabeza de clasificación. pesos: "imagenet" o None (desde cero)
    base = keras.applications.EfficientNetB0(include_top=False, weights=pesos, input_shape=FORMA_ENTRADA)
    base.trainable = base_entrenable

    entradas = keras.Input(shape=FORMA_ENTRADA, name="imagen")
    x = aumentacion(entradas) if aumentacion is not None else entradas
    # Con pesos preentrenados, BatchNormalization queda siempre en modo inferencia
    x = base(x, training=False) if pesos == "imagenet" else base(x)
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dropout(DROPOUT_CABEZA, name="dropout")(x)
    salidas = keras.layers.Dense(NUM_CLASES, activation="softmax", name="clasificador")(x)
    return keras.Model(entradas, salidas, name=nombre)


def obtener_backbone(modelo):
    # Se busca por nombre: en el experimento D también es un submodelo la capa de aumentación
    return next(capa for capa in modelo.layers if capa.name.startswith("efficientnet"))


def contar_parametros(modelo):
    entrenables = int(sum(np.prod(w.shape) for w in modelo.trainable_weights))
    return int(modelo.count_params()), entrenables


modelo_referencia = construir_modelo("referencia_tl")
backbone = obtener_backbone(modelo_referencia)
print(f"Backbone: {backbone.name} | capas: {len(backbone.layers)} | parámetros: {backbone.count_params():,}")
print(f"Salida del backbone: {backbone.output.shape}  (mapas de 7x7 con 1.280 canales)")
modelo_referencia.summary()
del modelo_referencia, backbone

## 3. Funciones comunes de entrenamiento y evaluación (c)

Los cuatro experimentos se entrenan con la misma función:

- **EarlyStopping** sobre `val_loss`: detiene el entrenamiento cuando la validación deja de mejorar y restaura los mejores pesos.
- **ModelCheckpoint**: guarda en `modelos/` el mejor modelo de cada experimento según `val_loss`.
- El historial se guarda en `resultados/history_<experimento>.csv`.
- Al terminar se cargan los pesos del *checkpoint* y el modelo se evalúa en validación y prueba.

Todos usan la pérdida `sparse_categorical_crossentropy` (etiquetas enteras y salida *softmax*) y el optimizador Adam. Lo que cambia entre experimentos es la tasa de aprendizaje y qué capas se entrenan.

In [ ]:
import time

modelos = {}
historiales = {}
resultados = {}


def compilar(modelo, tasa_aprendizaje):
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=tasa_aprendizaje),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )


def entrenar_experimento(clave, descripcion, modelo, archivo, epocas, paciencia, tasa_aprendizaje):
    ruta = MOD_DIR / archivo
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=paciencia, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(str(ruta), monitor="val_loss", save_best_only=True, verbose=0),
    ]
    inicio = time.time()
    history = modelo.fit(ds_train, validation_data=ds_val, epochs=epocas, callbacks=callbacks, verbose=2)
    duracion = time.time() - inicio

    historial = pd.DataFrame(history.history)
    historial.index = pd.RangeIndex(1, len(historial) + 1, name="época")
    historial.to_csv(RES_DIR / f"history_{clave}.csv")

    modelo.load_weights(ruta)  # mejor época según val_loss
    val_loss, val_acc = modelo.evaluate(ds_val, verbose=0)
    test_loss, test_acc = modelo.evaluate(ds_test, verbose=0)
    total, entrenables = contar_parametros(modelo)
    mejor_epoca = int(historial["val_loss"].idxmin())

    modelos[clave], historiales[clave] = modelo, historial
    resultados[clave] = {
        "descripcion": descripcion,
        "archivo": f"modelos/{archivo}",
        "tasa_aprendizaje": tasa_aprendizaje,
        "parametros_totales": total,
        "parametros_entrenables": entrenables,
        "epocas_entrenadas": len(historial),
        "mejor_epoca": mejor_epoca,
        "train_accuracy_mejor_epoca": float(historial.loc[mejor_epoca, "accuracy"]),
        "val_loss": float(val_loss),
        "val_accuracy": float(val_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "tiempo_entrenamiento_s": round(duracion, 1),
    }
    print(f"\n{descripcion}: val_accuracy = {val_acc:.4f} | test_accuracy = {test_acc:.4f} | "
          f"épocas = {len(historial)} (mejor: {mejor_epoca}) | tiempo = {duracion / 60:.1f} min")

## 4. Experimento A – CNN entrenada desde cero (línea base)

La misma EfficientNetB0, pero con **pesos aleatorios** (`weights=None`) y **todas las capas entrenables**. Como la arquitectura es idéntica a la de los demás experimentos, la diferencia entre A y B se debe únicamente a los pesos de ImageNet, lo que aísla el efecto del *Transfer Learning*.

Se espera un desempeño bajo: unas 3.100 imágenes de entrenamiento son muy pocas para aprender desde cero los filtros de una red con 4 millones de parámetros y distinguir 37 razas con diferencias sutiles.

| Hiperparámetro | Valor |
|---|---|
| Tasa de aprendizaje | 0,001 (Adam) |
| Épocas máximas | 30 |
| Paciencia de *early stopping* | 6 |

In [ ]:
EPOCAS_A, PACIENCIA_A, LR_A = 30, 6, 1e-3

modelo_A = construir_modelo("expA_cnn_scratch", pesos=None, base_entrenable=True)
compilar(modelo_A, LR_A)
entrenar_experimento("expA", "A – CNN desde cero", modelo_A, "t2_expA_cnn_scratch.keras", EPOCAS_A, PACIENCIA_A, LR_A)

## 5. Experimento B – Transfer Learning con backbone congelado (*feature extraction*)

EfficientNetB0 con **pesos de ImageNet** y **todas sus capas congeladas**. Funciona como un extractor de características fijo y solo se entrena la cabeza de clasificación (unos 47 mil parámetros para 37 clases). La idea es que los filtros aprendidos en ImageNet (bordes, texturas, formas, partes de animales) ya sirven para describir mascotas.

| Hiperparámetro | Valor |
|---|---|
| Tasa de aprendizaje | 0,001 (Adam) |
| Épocas máximas | 20 |
| Paciencia de *early stopping* | 4 |

In [ ]:
EPOCAS_B, PACIENCIA_B, LR_B = 20, 4, 1e-3

modelo_B = construir_modelo("expB_tl_frozen", pesos="imagenet", base_entrenable=False)
compilar(modelo_B, LR_B)
entrenar_experimento("expB", "B – TL congelado", modelo_B, "t2_expB_tl_frozen.keras", EPOCAS_B, PACIENCIA_B, LR_B)

## 6. Experimento C – Transfer Learning + fine-tuning

El *fine-tuning* parte del **mejor modelo del experimento B**: la cabeza ya está entrenada, así que sus gradientes iniciales son pequeños y no dañan los pesos preentrenados. Luego:

1. Se descongelan las **últimas 40 capas** del backbone, las más especializadas, que son las que más se benefician de adaptarse a razas de mascotas. Las primeras capas detectan bordes y texturas genéricas y se mantienen congeladas.
2. Las capas **BatchNormalization** siguen congeladas.
3. Se entrena con una **tasa de aprendizaje 100 veces menor (0,00001)**, para ajustar los pesos sin borrar lo aprendido en ImageNet.

| Hiperparámetro | Valor |
|---|---|
| Punto de partida | Mejor modelo del experimento B |
| Capas descongeladas | Últimas 40 del backbone (sin BatchNormalization) |
| Tasa de aprendizaje | 0,00001 (Adam) |
| Épocas máximas | 15 |
| Paciencia de *early stopping* | 4 |

El experimento D usa exactamente esta misma configuración. La única diferencia entre C y D es el Data Augmentation.

In [ ]:
EPOCAS_FT, PACIENCIA_FT, LR_FT = 15, 4, 1e-5
CAPAS_DESCONGELADAS = 40


def copiar_pesos(origen, destino):
    # Copia backbone y cabeza del modelo origen (mejor modelo de B) al modelo destino
    obtener_backbone(destino).set_weights(obtener_backbone(origen).get_weights())
    destino.get_layer("clasificador").set_weights(origen.get_layer("clasificador").get_weights())


def preparar_fine_tuning(modelo, n_capas=CAPAS_DESCONGELADAS):
    base = obtener_backbone(modelo)
    base.trainable = True
    for capa in base.layers[:-n_capas]:
        capa.trainable = False
    for capa in base.layers:
        if isinstance(capa, keras.layers.BatchNormalization):
            capa.trainable = False
    entrenables = sum(capa.trainable for capa in base.layers if capa.weights)
    total, entrenables_param = contar_parametros(modelo)
    print(f"Capas del backbone con pesos entrenables: {entrenables} de {sum(1 for c in base.layers if c.weights)} "
          f"| parámetros entrenables del modelo: {entrenables_param:,} de {total:,}")


modelo_C = construir_modelo("expC_tl_finetune", pesos="imagenet")
copiar_pesos(modelos["expB"], modelo_C)
preparar_fine_tuning(modelo_C)
compilar(modelo_C, LR_FT)
print("Accuracy de validación al iniciar (debe coincidir con B):", round(modelo_C.evaluate(ds_val, verbose=0)[1], 4))

entrenar_experimento("expC", "C – TL + fine-tuning", modelo_C, "t2_expC_tl_finetune.keras", EPOCAS_FT, PACIENCIA_FT, LR_FT)

## 7. Data Augmentation y Experimento D (d)

### 7.1 Capas de Data Augmentation

El Data Augmentation genera, en cada época, versiones distintas de las imágenes de entrenamiento con transformaciones que no cambian la raza. Así el modelo ve más variedad y memoriza menos.

| Capa | Transformación | Justificación |
|---|---|---|
| `RandomFlip("horizontal")` | Espejo horizontal | Una mascota mirando a la izquierda o a la derecha es la misma raza |
| `RandomRotation(0.1)` | Rotación de hasta ±36° | Fotos tomadas con la cámara inclinada |
| `RandomZoom(0.15)` | Acercamiento o alejamiento de hasta 15 % | Distintas distancias y encuadres |
| `RandomContrast(0.2)` | Contraste ± 20 % | Distintas condiciones de iluminación |

Las capas van **dentro del modelo**, justo después de la entrada. Keras solo las activa durante `fit` (entrenamiento) y las desactiva automáticamente en validación, evaluación y predicción. Por eso el aumento se aplica **solo a los datos de entrenamiento**.

No se usa el volteo vertical: una mascota cabeza abajo es una imagen irreal para este problema.

In [ ]:
aumentacion = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal", seed=SEMILLA),
        keras.layers.RandomRotation(0.1, seed=SEMILLA),
        keras.layers.RandomZoom(0.15, seed=SEMILLA),
        keras.layers.RandomContrast(0.2, seed=SEMILLA),  # opera sobre el rango [0, 255] que recibe el modelo
    ],
    name="aumentacion",
)

# Visualización del efecto sobre una imagen de entrenamiento
imagen_ejemplo = next(iter(ds_train))[0][:1]
fig, axes = plt.subplots(3, 4, figsize=(12, 9.5))
for i, ax in enumerate(axes.flat):
    vista = imagen_ejemplo if i == 0 else aumentacion(imagen_ejemplo, training=True)
    ax.imshow(np.clip(vista[0].numpy(), 0, 255).astype("uint8"))
    ax.set_title("Original" if i == 0 else f"Aumentada {i}", fontsize=11, fontweight="bold" if i == 0 else None)
    ax.axis("off")
fig.suptitle("Efecto del Data Augmentation sobre una misma imagen", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_data_augmentation.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Experimento D – Transfer Learning + fine-tuning + Data Augmentation

Misma configuración que el experimento C: parte del mejor modelo de B, descongela las últimas 40 capas y usa una tasa de aprendizaje de 0,00001 durante máximo 15 épocas. La única diferencia es la capa de aumentación al inicio del modelo. Así, la comparación C vs. D mide solamente el efecto del Data Augmentation.

In [ ]:
modelo_D = construir_modelo("expD_tl_finetune_da", pesos="imagenet", aumentacion=aumentacion)
copiar_pesos(modelos["expB"], modelo_D)
preparar_fine_tuning(modelo_D)
compilar(modelo_D, LR_FT)
print("Accuracy de validación al iniciar (debe coincidir con B):", round(modelo_D.evaluate(ds_val, verbose=0)[1], 4))

entrenar_experimento("expD", "D – TL + fine-tuning + DA", modelo_D, "t2_expD_tl_finetune_da.keras", EPOCAS_FT, PACIENCIA_FT, LR_FT)

## 8. Comparación de experimentos (e)

### 8.1 Tabla comparativa

La columna **Brecha train−val** es la diferencia entre la accuracy de entrenamiento y la de validación en la mejor época; sirve como indicador de sobreajuste. En el experimento D, la accuracy de entrenamiento se mide sobre imágenes aumentadas, que son más difíciles, así que su brecha tiende a ser menor.

In [ ]:
tabla = pd.DataFrame(resultados).T
tabla["brecha_train_val"] = tabla["train_accuracy_mejor_epoca"] - tabla["val_accuracy"]
tabla_comparativa = tabla[
    ["descripcion", "test_accuracy", "test_loss", "val_accuracy", "brecha_train_val",
     "parametros_totales", "parametros_entrenables", "epocas_entrenadas", "mejor_epoca",
     "tasa_aprendizaje", "tiempo_entrenamiento_s"]
].rename(columns={
    "descripcion": "Experimento", "test_accuracy": "Accuracy test", "test_loss": "Loss test",
    "val_accuracy": "Accuracy val", "brecha_train_val": "Brecha train-val",
    "parametros_totales": "Parámetros totales", "parametros_entrenables": "Parámetros entrenables",
    "epocas_entrenadas": "Épocas", "mejor_epoca": "Mejor época", "tasa_aprendizaje": "Tasa de aprendizaje",
    "tiempo_entrenamiento_s": "Tiempo (s)",
})
tabla_comparativa.index.name = "Clave"
tabla_comparativa.to_csv(RES_DIR / "tabla_comparativa.csv", encoding="utf-8")
# Copia con formato legible solo para mostrar (el CSV conserva los valores numéricos)
vista = tabla_comparativa.copy()
formatos = {"Accuracy test": "{:.2%}", "Loss test": "{:.4f}", "Accuracy val": "{:.2%}", "Brecha train-val": "{:+.2%}",
            "Parámetros totales": "{:,}", "Parámetros entrenables": "{:,}", "Tasa de aprendizaje": "{:g}"}
for columna, formato in formatos.items():
    vista[columna] = vista[columna].map(formato.format)
display(vista)

### 8.2 Curvas de entrenamiento

En C y D las épocas corresponden solo a la fase de *fine-tuning*, que continúa desde el mejor punto de B.

In [ ]:
import matplotlib.ticker

claves = list(resultados)
fig, axes = plt.subplots(2, len(claves), figsize=(5 * len(claves), 8.5), sharey="row")
for j, clave in enumerate(claves):
    historial = historiales[clave]
    mejor = resultados[clave]["mejor_epoca"]
    for i, (metrica, nombre) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")]):
        ax = axes[i, j]
        ax.plot(historial.index, historial[metrica], marker="o", markersize=3, label="Entrenamiento")
        ax.plot(historial.index, historial[f"val_{metrica}"], marker="o", markersize=3, label="Validación")
        ax.axvline(mejor, color="gray", linestyle="--", label=f"Mejor época ({mejor})")
        ax.set_title(f"{resultados[clave]['descripcion']}\n{nombre}" if i == 0 else nombre, fontsize=11)
        ax.set_xlabel("Época")
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
        if j == 0:
            ax.set_ylabel(nombre)
        ax.legend(fontsize=8)
fig.suptitle("Curvas de entrenamiento de los cuatro experimentos", fontsize=15)
fig.tight_layout()
fig.savefig(FIG_DIR / "04_curvas_entrenamiento.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.3 Accuracy de prueba por experimento

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
nombres = [resultados[c]["descripcion"] for c in claves]
valores = [resultados[c]["test_accuracy"] for c in claves]
barras = ax.bar(nombres, valores, color=["#9e9e9e", "#4c78a8", "#f58518", "#54a24b"][: len(claves)])
ax.axhline(META_ACCURACY, color="red", linestyle="--", label=f"Meta ({META_ACCURACY:.0%})")
for barra, valor in zip(barras, valores):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.01, f"{valor:.2%}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.08)
ax.set_ylabel("Accuracy en prueba")
ax.set_title("Comparación de experimentos – accuracy en el conjunto de prueba")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_comparacion_experimentos.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.4 Mejor modelo: matriz de confusión y reporte por clase

El mejor experimento se elige por **accuracy de validación**, no de prueba. El conjunto de prueba debe quedar fuera de toda decisión; si se usara para elegir el modelo, su accuracy dejaría de ser una estimación honesta del desempeño con datos nuevos.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

MEJOR = max(resultados, key=lambda c: resultados[c]["val_accuracy"])
modelo_mejor = modelos[MEJOR]
print(f"Mejor experimento (por validación): {resultados[MEJOR]['descripcion']}")
print(f"Accuracy en prueba: {resultados[MEJOR]['test_accuracy']:.4f}")

probabilidades = modelo_mejor.predict(ds_test, verbose=0)
y_pred = probabilidades.argmax(axis=1)
confianza = probabilidades.max(axis=1)
print(f"Accuracy a partir de las predicciones: {(y_pred == y_test).mean():.4f}")

matriz = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASES))
lado = max(7, 0.42 * NUM_CLASES)
fig, ax = plt.subplots(figsize=(lado + 2, lado))
sns.heatmap(
    matriz, annot=True, fmt="d", cmap="Blues", linewidths=0.3, linecolor="white",
    xticklabels=ETIQUETAS_LEGIBLES, yticklabels=ETIQUETAS_LEGIBLES,
    annot_kws={"size": 6 if NUM_CLASES > 10 else 14}, cbar_kws={"label": "Número de imágenes"}, ax=ax,
)
ax.set_xlabel("Clase predicha")
ax.set_ylabel("Clase real")
ax.set_title(f"Matriz de confusión – {resultados[MEJOR]['descripcion']} (accuracy = {resultados[MEJOR]['test_accuracy']:.2%})")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=8 if NUM_CLASES > 10 else 12)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8 if NUM_CLASES > 10 else 12)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_matriz_confusion_mejor.png", dpi=150, bbox_inches="tight")
plt.show()

reporte = classification_report(y_test, y_pred, labels=np.arange(NUM_CLASES), target_names=ETIQUETAS_LEGIBLES, digits=4)
print(reporte)
(RES_DIR / "classification_report_mejor.txt").write_text(
    f"Taller 2 – {resultados[MEJOR]['descripcion']}: reporte de clasificación (conjunto de prueba)\n\n" + reporte,
    encoding="utf-8",
)

### 8.5 Clases que más se confunden

In [ ]:
errores_matriz = matriz.copy()
np.fill_diagonal(errores_matriz, 0)
pares = pd.DataFrame(
    [
        (ETIQUETAS_LEGIBLES[r], ETIQUETAS_LEGIBLES[p], errores_matriz[r, p])
        for r in range(NUM_CLASES) for p in range(NUM_CLASES) if errores_matriz[r, p] > 0
    ],
    columns=["Real", "Predicho", "Errores"],
).sort_values("Errores", ascending=False).head(12).reset_index(drop=True)
print("Confusiones más frecuentes (real → predicho):")
display(pares)

por_clase = pd.DataFrame(
    {"Muestras": matriz.sum(axis=1), "Aciertos": np.diag(matriz)}, index=pd.Index(ETIQUETAS_LEGIBLES, name="Clase")
)
por_clase["Errores"] = por_clase["Muestras"] - por_clase["Aciertos"]
por_clase["Accuracy (%)"] = (por_clase["Aciertos"] / por_clase["Muestras"] * 100).round(2)
print("Clases con menor accuracy:")
display(por_clase.sort_values("Accuracy (%)").head(10))

# Errores entre especies (un perro clasificado como gato o al revés) cuando la tarea es por razas
if OBJETIVO == "label":
    especie_de_raza = pd.Series(etiquetas_test[:, 1], index=etiquetas_test[:, 0]).groupby(level=0).first()
    cruzados = int((especie_de_raza.loc[y_pred].values != especie_de_raza.loc[y_test].values).sum())
    print(f"Errores entre especies: {cruzados} de {int((y_pred != y_test).sum())} errores totales")

## 9. Análisis técnico (f)

La siguiente celda calcula el aporte de cada técnica en puntos porcentuales de accuracy de prueba. El análisis se apoya en esas cifras y en la tabla y figuras de la sección 8.

In [ ]:
acc = {clave: resultados[clave]["test_accuracy"] for clave in resultados}
efectos = pd.DataFrame(
    [
        ("Transfer Learning", "A → B", acc["expB"] - acc["expA"]),
        ("Fine-tuning", "B → C", acc["expC"] - acc["expB"]),
        ("Data Augmentation", "C → D", acc["expD"] - acc["expC"]),
        ("Total", f"A → {MEJOR[-1]}", acc[MEJOR] - acc["expA"]),
    ],
    columns=["Técnica", "Comparación", "Cambio en accuracy de prueba"],
)
display(efectos.assign(**{"Cambio en accuracy de prueba": efectos["Cambio en accuracy de prueba"].map("{:+.2%}".format)}))
print(f"Mejor experimento: {resultados[MEJOR]['descripcion']} → {acc[MEJOR]:.2%} en prueba "
      f"({'CUMPLE' if acc[MEJOR] >= META_ACCURACY else 'NO CUMPLE'} la meta del {META_ACCURACY:.0%})")

### Efecto del Transfer Learning (A vs. B y C)

Con los mismos datos y la misma arquitectura, el modelo entrenado desde cero (A) tiene que aprender todos sus filtros a partir de unas 3.100 imágenes, demasiado pocas para una red de 4 millones de parámetros y 37 clases con diferencias sutiles (pelaje, forma de las orejas, hocico). Sus curvas muestran el resultado: el modelo aprende muy despacio y, con tan pocos ejemplos, lo poco que aprende está ligado a las imágenes de entrenamiento y no se traslada bien a validación.

El experimento B reutiliza filtros aprendidos con 1,2 millones de imágenes de ImageNet, que ya distinguen texturas de pelo, formas de cabeza y partes de animales. Entrenando solo la cabeza, del orden de 47 mil parámetros para 37 clases frente a los más de 4 millones de A, el problema pasa de aprender a ver a aprender a combinar lo que ya se ve. Por eso el salto de A a B es, con diferencia, la mejora más grande del taller.

Hay un factor que conviene reconocer: ImageNet incluye muchas razas de perros y gatos entre sus 1.000 clases. Por eso los filtros preentrenados encajan especialmente bien en este dataset, y la ventaja del *Transfer Learning* es mayor de lo que sería en un dominio muy distinto (por ejemplo, imágenes médicas).

### Efecto del fine-tuning (B vs. C)

El *fine-tuning* ajusta las últimas capas del backbone, las que codifican características más específicas, para que se especialicen en las diferencias entre razas. Como se parte de un modelo que ya funciona bien, la mejora esperada es moderada. La tasa de aprendizaje baja (0,00001) y las capas BatchNormalization congeladas evitan que se destruya lo aprendido. Si las curvas de C muestran que la accuracy de entrenamiento sube mucho más que la de validación, el modelo empieza a memorizar: con pocos datos, descongelar más capas aumenta ese riesgo.

### Efecto del Data Augmentation (C vs. D)

El Data Augmentation actúa como regularizador: en cada época el modelo ve versiones distintas de cada foto, así que le cuesta más memorizar. Su efecto se observa sobre todo en la **brecha entre entrenamiento y validación**, que debe ser menor en D. En accuracy de prueba, el efecto puede ser pequeño o incluso nulo en pocas épocas de *fine-tuning*: partir de un modelo ya preentrenado y regularizado deja poco margen, y el DA suele necesitar más épocas para rendir. La comparación es justa porque C y D parten del mismo punto y solo difieren en esa capa.

### Accuracy alcanzada frente al 90 % exigido

La tabla de efectos indica si el mejor experimento, elegido por validación, supera la meta en prueba. En clasificación de 37 razas con unas 85 imágenes de entrenamiento por clase, superar el 90 % solo es posible con *Transfer Learning*: el modelo desde cero queda muy lejos de la meta.

### Razas que más se confunden

Las tablas de la sección 8.5 muestran las parejas concretas. En este dataset, los errores se concentran típicamente en razas **de la misma especie y con aspecto parecido**, como el American Pit Bull Terrier y el Staffordshire Bull Terrier (prácticamente indistinguibles en foto), el British Shorthair y el Russian Blue (gatos grises de pelo corto), o el Ragdoll y el Birman (gatos de pelo largo con puntas oscuras). La sección 8.5 también cuenta los errores entre especies (perro ↔ gato). Si son muy pocos frente al total, el modelo separa bien perros y gatos y la dificultad está en los detalles finos entre razas parecidas.

### Limitaciones

1. **Pocos datos por clase:** unas 85 imágenes de entrenamiento por raza limitan cuánto se puede ajustar el backbone sin sobreajustar.
2. **Redimensionado sin conservar la proporción:** llevar todas las imágenes a 224 × 224 deforma ligeramente a las mascotas.
3. **Fondos y poses variados:** la mascota puede ocupar una parte pequeña de la imagen o aparecer parcialmente, y el modelo no recibe información de dónde está.
4. **Razas casi idénticas visualmente:** algunas parejas se distinguen por rasgos que una foto no siempre muestra.
5. **Validación pequeña** (unas 550 imágenes, ~15 por raza): la diferencia entre experimentos cercanos puede estar dentro del margen de variación aleatoria.

### Mejoras posibles

- **Backbones más grandes o modernos** (EfficientNetB3–B7, EfficientNetV2, ConvNeXt), a mayor resolución.
- **Fine-tuning gradual:** descongelar más bloques por etapas, con una tasa de aprendizaje decreciente (*cosine decay*).
- **Más épocas con Data Augmentation** y técnicas más fuertes (*RandAugment*, *MixUp*, *CutMix*).
- **Recortar la región de la mascota** usando el recuadro de la cabeza o la máscara de segmentación del propio dataset, que conecta directamente con el proyecto final de segmentación.
- **Validación cruzada** o varias semillas, para medir la variabilidad de los resultados.

## 10. Resumen de métricas y red de seguridad

Se exportan las métricas de los cuatro experimentos a `resultados/metricas_taller2.json` y se comprueba la meta. Si no se alcanza el 90 %, la celda indica cómo activar la red de seguridad (clasificación perro / gato).

In [ ]:
import json

metricas = {
    "dataset": "Oxford-IIIT Pet",
    "objetivo": OBJETIVO,
    "num_clases": NUM_CLASES,
    "conjuntos": {"entrenamiento": int(len(y_train_np)), "validacion": int(len(y_val_np)), "prueba": int(len(y_test_np))},
    "backbone": "EfficientNetB0 (ImageNet)",
    "configuracion": {
        "tam_imagen": TAM_IMAGEN, "batch_size": TAMANO_BATCH, "dropout_cabeza": DROPOUT_CABEZA,
        "capas_descongeladas_fine_tuning": CAPAS_DESCONGELADAS, "semilla": SEMILLA,
    },
    "experimentos": resultados,
    "mejor_experimento": MEJOR,
    "criterio_seleccion": "mayor accuracy de validación",
    "mejor_test_accuracy": resultados[MEJOR]["test_accuracy"],
    "meta_accuracy": META_ACCURACY,
    "cumple_meta": bool(resultados[MEJOR]["test_accuracy"] >= META_ACCURACY),
}
(RES_DIR / "metricas_taller2.json").write_text(json.dumps(metricas, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(metricas, ensure_ascii=False, indent=2))

if metricas["cumple_meta"]:
    print(f"\nMeta cumplida con {resultados[MEJOR]['descripcion']}: {metricas['mejor_test_accuracy']:.2%} en prueba.")
elif OBJETIVO == "label":
    print("\nNo se alcanzó la meta con 37 razas. Red de seguridad: cambiar OBJETIVO = \"species\" "
          "en la celda 1.1 y ejecutar todo de nuevo (Entorno de ejecución > Ejecutar todas).")